In [14]:
import warnings
import numpy as np
import pandas as pd

# ─── Silence noisy warnings ─────────────────────────────────────────────────────
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=FutureWarning)                   # pandas downcast warning
warnings.filterwarnings("ignore", category=ConvergenceWarning)              # IterativeImputer & MLP
warnings.filterwarnings("ignore", message=".*valid feature names.*")        # LGBM warning
warnings.filterwarnings("ignore", message=".*use_label_encoder.*")          # XGBoost warning

from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# ─── 1) Load ─────────────────────────────────────────────────────────────────────
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

# ─── 2) Feature engineering ─────────────────────────────────────────────────────
def process(df):
    df['Deck']      = df['Cabin'].str.split('/').str[0].fillna('F')
    df['Side']      = df['Cabin'].str.split('/').str[2].fillna('P')
    df['Group']     = df['PassengerId'].str.split('_').str[0]
    df['GroupSize'] = df.groupby('Group')['PassengerId'].transform('count')
    df['Alone']     = (df['GroupSize']==1).astype(int)

    spend_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
    df['TotalSpend']   = df[spend_cols].sum(axis=1)
    df['ZeroSpender']  = (df['TotalSpend']==0).astype(int)
    df['TotalSpend']   = np.log1p(df['TotalSpend'])

    df['CryoSleep'] = (
        df['CryoSleep']
          .fillna(df['ZeroSpender'].map({1: True, 0: pd.NA}))
          .astype('boolean')
          .fillna(False)
    )
    df['VIP'] = (
        df['VIP']
          .astype('boolean')
          .fillna(False)
    )

    df['HomePlanet']  = df['HomePlanet'].fillna('Earth')
    df['Destination'] = df['Destination'].fillna('TRAPPIST-1e')
    return df

train = process(train)
test  = process(test)

# 3) Prepare X, y
drop_cols = ['PassengerId','Name','Cabin','Transported','Group']
X      = train.drop(columns=drop_cols)
y      = train['Transported'].astype(int)

# explicitly drop only the columns that exist in test
X_test = test.drop(columns=['PassengerId','Name','Cabin','Group'])


# ─── 4) Column lists ────────────────────────────────────────────────────────────
categorical = X.select_dtypes(include="object").columns.tolist()
numerical   = X.select_dtypes(include=["int64","float64","boolean","bool"]).columns.tolist()

# ─── 5) Pipelines ───────────────────────────────────────────────────────────────
num_imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=50, random_state=42),
    max_iter=10,
    random_state=42
)
numerical_pipeline = Pipeline([
    ('imputer', num_imputer),
    ('scaler',  StandardScaler())
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline,   numerical),
    ('cat', categorical_pipeline, categorical)
])

# ─── 6) Models ─────────────────────────────────────────────────────────────────
xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    n_estimators=465,
    learning_rate=0.04974445885593547,
    max_depth=4,
    subsample=0.7023223983232721,
    colsample_bytree=0.8807540583107164,
    min_child_weight=1,
    gamma=0.3091801139027462
)

lgb = LGBMClassifier(
    random_state=42,
    n_estimators=337,
    learning_rate=0.024230875709953033,
    max_depth=9,
    num_leaves=26,
    subsample=0.9242197257138703,
    colsample_bytree=0.6030490088823314,
    min_child_samples=18,
    verbose=-1
)

rf = RandomForestClassifier(
    random_state=42,
    n_estimators=192,
    max_depth=11,
    min_samples_split=7,
    min_samples_leaf=7,
    max_features='sqrt'
)

mlp_base = MLPClassifier(
    random_state=42,
    hidden_layer_sizes=(64,),
    activation='relu',
    alpha=0.002444950220293307,
    learning_rate_init=0.00018999966184724145,
    max_iter=500
)
mlp = CalibratedClassifierCV(mlp_base, method='sigmoid', cv=3)

models = {"xgb": xgb, "lgb": lgb, "rf": rf, "mlp": mlp}

# ─── 7) 5-fold CV + threshold tuning + test-proba aggregation ────────────────
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
test_preds      = np.zeros((X_test.shape[0], len(models)))
val_scores      = {m: [] for m in models}
best_thresholds = {m: [] for m in models}

for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    X_tr_prep   = preprocessor.fit_transform(X_tr)
    X_val_prep  = preprocessor.transform(X_val)
    X_test_prep = preprocessor.transform(X_test)

    for i, (name, model) in enumerate(models.items()):
        model.fit(X_tr_prep, y_tr)
        proba_val = model.predict_proba(X_val_prep)[:,1]

        # threshold search
        best_thr, best_acc = 0.5, 0
        for thr in np.arange(0.3, 0.7, 0.01):
            acc = accuracy_score(y_val, (proba_val >= thr).astype(int))
            if acc > best_acc:
                best_acc, best_thr = acc, thr

        val_scores[name].append(best_acc)
        best_thresholds[name].append(best_thr)
        print(f"Fold {fold} | {name:>3} | Acc: {best_acc:.4f} @thr={best_thr:.2f}")

        test_preds[:, i] += model.predict_proba(X_test_prep)[:,1] / kf.n_splits

# ─── 8) CV summary ─────────────────────────────────────────────────────────────
for name in models:
    mean_acc = np.mean(val_scores[name])
    mean_thr = np.mean(best_thresholds[name])
    print(f"{name:>3} Mean CV Acc: {mean_acc:.4f}; Mean Thr: {mean_thr:.2f}")

# ─── 9) Ensemble & save ───────────────────────────────────────────────────────
weights = {m: np.mean(accs) for m, accs in val_scores.items()}
total   = sum(weights.values())
weights = {m: w/total for m, w in weights.items()}

final_score = np.zeros(X_test.shape[0])
for i, m in enumerate(models):
    thr = np.mean(best_thresholds[m])
    final_score += weights[m] * ((test_preds[:,i] >= thr).astype(int))

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": final_score >= 0.5
})
submission.to_csv("submission.csv", index=False)
print("Done! Submission saved.")

Fold 1 | xgb | Acc: 0.8148 @thr=0.48
Fold 1 | lgb | Acc: 0.8166 @thr=0.49
Fold 1 |  rf | Acc: 0.8148 @thr=0.44
Fold 1 | mlp | Acc: 0.8085 @thr=0.52
Fold 2 | xgb | Acc: 0.8028 @thr=0.46
Fold 2 | lgb | Acc: 0.8005 @thr=0.47
Fold 2 |  rf | Acc: 0.7947 @thr=0.56
Fold 2 | mlp | Acc: 0.8028 @thr=0.48
Fold 3 | xgb | Acc: 0.8114 @thr=0.43
Fold 3 | lgb | Acc: 0.8137 @thr=0.52
Fold 3 |  rf | Acc: 0.8074 @thr=0.48
Fold 3 | mlp | Acc: 0.8125 @thr=0.50
Fold 4 | xgb | Acc: 0.8251 @thr=0.49
Fold 4 | lgb | Acc: 0.8182 @thr=0.52
Fold 4 |  rf | Acc: 0.8084 @thr=0.49
Fold 4 | mlp | Acc: 0.8165 @thr=0.45
Fold 5 | xgb | Acc: 0.7992 @thr=0.54
Fold 5 | lgb | Acc: 0.7969 @thr=0.55
Fold 5 |  rf | Acc: 0.7980 @thr=0.50
Fold 5 | mlp | Acc: 0.7957 @thr=0.58
xgb Mean CV Acc: 0.8107; Mean Thr: 0.48
lgb Mean CV Acc: 0.8092; Mean Thr: 0.51
 rf Mean CV Acc: 0.8047; Mean Thr: 0.49
mlp Mean CV Acc: 0.8072; Mean Thr: 0.51
Done! Submission saved.
